# Capítulo 3: Variáveis Aleatórias Discretas

Notebook com o **código** deste capítulo, para o Google Colab. Cada trecho vem precedido de uma explicação curta; o texto completo está no site do livro.

Rode a célula de **setup** abaixo primeiro (uma vez), depois as demais em ordem. A seção 3.7 (Poisson) é leitura complementar e não entra aqui.

In [ ]:
# Setup (rode uma vez).
!curl -sO https://raw.githubusercontent.com/BragaD/UnDF-Bases3-Estatistica-202602/main/formato.py   # baixa o ajudante de formatação do livro

Importa as bibliotecas usadas no capítulo. O `set_printoptions` faz o numpy mostrar números sem o prefixo `np.float64`.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from itertools import product, permutations
from math import comb
from scipy import stats
from formato import num

np.set_printoptions(legacy="1.25")

## 3.1 Variável Aleatória e Função de Probabilidade

Monta o espaço amostral dos dois módulos (pronto, com retrabalho ou inutilizável), com a probabilidade de cada par e o lucro de cada um, em milhares de reais.

In [ ]:
prob_a = {"P": 0.80, "R": 0.10, "I": 0.10}
prob_b = {"P": 0.70, "R": 0.20, "I": 0.10}

def lucro(a, b):
    if "I" in (a, b):
        return 5 - 10                          # versão reduzida
    retrabalho = 5 * ((a == "R") + (b == "R"))
    return 25 - 10 - retrabalho

linhas = []
for a, b in product("PRI", repeat=2):
    linhas.append((a + b, prob_a[a] * prob_b[b], lucro(a, b)))

omega = pd.DataFrame(linhas, columns=["resultado", "probabilidade", "lucro"])
omega.round(2)

Função de probabilidade do lucro: junta os pontos com o mesmo lucro e soma as probabilidades.

In [ ]:
fp = omega.groupby("lucro")["probabilidade"].sum().sort_index(ascending=False)
fp.round(2)

Gráfico de hastes da função de probabilidade do lucro.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 3.5))
ax.vlines(fp.index, 0, fp.values, linewidth=3)
ax.plot(fp.index, fp.values, "o")
ax.set_xticks(fp.index)
ax.set_xlabel("lucro x (R$ mil)")
ax.set_ylabel("p(x)")
ax.set_ylim(0, 0.65)
plt.show()

Outra variável no mesmo experimento: o custo de retrabalho.

In [ ]:
def retrabalho(a, b):
    if "I" in (a, b):
        return 0
    return 5 * ((a == "R") + (b == "R"))

omega["retrabalho"] = [retrabalho(r[0], r[1]) for r in omega["resultado"]]
omega.groupby("retrabalho")["probabilidade"].sum().round(2)

Número de PRs com bug em dois sorteios sem reposição, em contagens sobre os 20 pares ordenados.

In [ ]:
from itertools import permutations

prs = ["B1", "B2", "S1", "S2", "S3"]
pares = list(permutations(prs, 2))

bugs = pd.Series([sum(p.startswith("B") for p in par) for par in pares])
bugs.value_counts().sort_index()

## 3.2 Valor Médio e Variância

Valores e probabilidades do lucro por sistema vendido.

In [ ]:
x = np.array([15, 10, 5, -5])
p = np.array([0.56, 0.23, 0.02, 0.19])

Valor médio do lucro.

In [ ]:
E = (x * p).sum()
round(E, 2)

Simula 10.000 vendas com semente 42: a média das vendas fica perto do valor médio.

In [ ]:
rng = np.random.default_rng(42)
vendas = rng.choice(x, size=10_000, p=p)
round(float(vendas.mean()), 2)

Variância e desvio-padrão do lucro.

In [ ]:
Var = ((x - E) ** 2 * p).sum()
DP = np.sqrt(Var)
round(Var, 2), round(DP, 2)

Preços e custos em dobro: o valor médio dobra e a variância quadruplica.

In [ ]:
z = 2 * x
E_z = (z * p).sum()
Var_z = ((z - E_z) ** 2 * p).sum()
round(E_z, 2), round(Var_z, 2)

Taxa fixa de 3 mil: o valor médio desce 3 e a variância não muda.

In [ ]:
w = x - 3
E_w = (w * p).sum()
Var_w = ((w - E_w) ** 2 * p).sum()
round(E_w, 2), round(Var_w, 2)

## 3.3 Função de Distribuição Acumulada

Os mesmos valores em ordem crescente, para acumular da esquerda para a direita.

In [ ]:
x = np.array([-5, 5, 10, 15])
p = np.array([0.19, 0.02, 0.23, 0.56])

Tabela com a função de probabilidade e a f.d.a., feita com `np.cumsum`.

In [ ]:
fda = pd.DataFrame({"x": x, "p(x)": p, "F(x)": np.cumsum(p)})
fda.round(2)

Gráfico em escada da f.d.a., com pontos fechados e abertos nos degraus.

In [ ]:
F = np.cumsum(p)
bordas = np.concatenate(([-10], x, [20]))
alturas = np.concatenate(([0], F))

fig, ax = plt.subplots(figsize=(6, 3.5))
ax.hlines(alturas, bordas[:-1], bordas[1:], linewidth=2)
ax.plot(x, F, "o", color="C0")                        # ponto fechado: F(x) inclui x
ax.plot(x, alturas[:-1], "o", mfc="white", color="C0") # ponto aberto: valor antes do degrau
ax.set_xlabel("lucro x (R$ mil)")
ax.set_ylabel("F(x)")
plt.show()

Probabilidades de intervalos como diferenças de valores da f.d.a.

In [ ]:
def F_lucro(t):
    return p[x <= t].sum()

round(F_lucro(10) - F_lucro(0), 2), round(1 - F_lucro(5), 2), round(F_lucro(7.3), 2)

Compara a f.d.a. com a proporção acumulada em 10.000 vendas simuladas.

In [ ]:
rng = np.random.default_rng(42)
vendas = rng.choice(x[::-1], size=10_000, p=p[::-1])   # mesma ordem da seção 3.2

empirica = [(vendas <= v).mean() for v in x]
pd.DataFrame({"x": x, "F(x)": np.cumsum(p), "proporção nas vendas": empirica}).round(3)

## 3.4 Uniforme Discreta e Bernoulli

Uniforme discreta com `stats.randint`: o limite superior fica de fora.

In [ ]:
dado = stats.randint(1, 7)          # valores 1, 2, ..., 6 (o 7 fica de fora)
servidor = stats.randint(0, 10)     # valores 0, 1, ..., 9

print(round(dado.pmf(6), 4), dado.mean(), round(dado.var(), 4))
print(servidor.mean(), servidor.var())

Bernoulli com `stats.bernoulli`: requisição que falha com probabilidade 0,05.

In [ ]:
falha = stats.bernoulli(0.05)
falha.pmf([0, 1]), falha.mean(), falha.var()

Variância da Bernoulli em função de p: máxima em p = 0,5.

In [ ]:
p_grade = np.linspace(0, 1, 201)

fig, ax = plt.subplots(figsize=(6, 3.5))
ax.plot(p_grade, p_grade * (1 - p_grade))
ax.axvline(0.5, color="gray", linestyle="--", linewidth=1)
ax.set_xlabel("p")
ax.set_ylabel("Var(X) = p(1 − p)")
plt.show()

## 3.5 Distribuição Binomial

Probabilidades binomiais pela fórmula, com `math.comb`, para 10 jobs com p = 0,1.

In [ ]:
n, p = 10, 0.1
[round(comb(n, k) * p**k * (1 - p)**(n - k), 4) for k in range(4)]

As mesmas probabilidades com `stats.binom`: `pmf` e `cdf`.

In [ ]:
jobs = stats.binom(n, p)
tabela = pd.DataFrame({"k": range(11), "P(X = k)": jobs.pmf(range(11)), "P(X ≤ k)": jobs.cdf(range(11))})
tabela.head(5).round(4)

Gráfico da distribuição do número de falhas.

In [ ]:
k = np.arange(11)
fig, ax = plt.subplots(figsize=(6, 3.5))
ax.vlines(k, 0, jobs.pmf(k), linewidth=3)
ax.plot(k, jobs.pmf(k), "o")
ax.set_xticks(k)
ax.set_xlabel("número de falhas k")
ax.set_ylabel("P(X = k)")
plt.show()

Simula 10.000 noites somando 10 Bernoullis e compara média e variância com np e np(1 − p).

In [ ]:
rng = np.random.default_rng(42)
ensaios = rng.random((10_000, n)) < p          # 10.000 noites x 10 jobs: True = falha
noites = ensaios.sum(axis=1)                    # número de falhas em cada noite

round(float(noites.mean()), 3), round(float(noites.var()), 3), jobs.mean(), jobs.var()

Probabilidade de pelo menos 80 sucessos em 100 requisições, pelo complementar.

In [ ]:
requisicoes = stats.binom(100, 0.9)
round(1 - requisicoes.cdf(79), 4), requisicoes.mean(), requisicoes.std()

## 3.6 Distribuição Hipergeométrica

A auditoria de PRs como hipergeométrica, com os nomes dos argumentos da scipy.

In [ ]:
# scipy.stats.hypergeom(M, n, N): M = tamanho da população (nosso N),
# n = número de sucessos na população (nosso r), N = tamanho da amostra (nosso n)
auditoria = stats.hypergeom(M=20, n=5, N=4)

pd.DataFrame({"k": range(5), "P(X = k)": auditoria.pmf(range(5))}).round(4)

Valor médio e variância, comparados com a binomial.

In [ ]:
p_bug = 5 / 20
auditoria.mean(), round(auditoria.var(), 4), round(4 * p_bug * (1 - p_bug), 4)

Com populações cada vez maiores, a hipergeométrica se aproxima da binomial.

In [ ]:
tamanhos = [20, 100, 1_000, 10_000]
comparacao = pd.DataFrame({
    "N": tamanhos,
    "hipergeométrica": [stats.hypergeom(M=N, n=N // 10, N=5).pmf(0) for N in tamanhos],
    "binomial b(5, 1/10)": stats.binom(5, 0.1).pmf(0),
})
comparacao.round(4)